---
layout: article
title: Rent Vs. Purchase 
custom_css: article.css
include_mathjax: true
publish: false
---

## Introduction ##

If you possess a specific amount of Cash ($W$), how can you better improve your position when you are faced with the following two choices:

+ Rent an apartment
+ Purchase a small house

In this article, we compare these two options by analyzing the "time" your wealth can buy in each scenario and the subsequent impact on the overall position.

## Basic Calculation for $\Delta N$ ##

To compare these options effectively, we calculate the duration for which the cash lasts in each scenario and determine the difference ($\Delta N$). We adopt the equations developed by [Diogenes Analytics](https://diogenesanalytics.com/blog/2024/05/12/rent-or-buy).

The duration of wealth when renting ($N_{\text rent}$) and purchasing ($N_{\text purchase}$) are defined as follows:

$$
N_{\text rent} = \frac{W}{C+R}
$$

$$
N_{\text purchase} = \frac{(1-p)W}{(C-H)}
$$

Consequently, the difference in time ($\Delta N$) is calculated as:

$$
\Delta N = N_{\text purchase} - N_{\text rent}
= \frac{(1-p)W}{(C-H)} - \frac{W}{(C+R)}
$$

Where:
+ $N_{\text purchase}$: Number of years the remaining wealth can purchase after the property investment. 
+ $N_{\text rent}$: Number of years the original wealth can purchase

+ $p$: Percentage of wealth used to purchase a house and rennovate it
+ $W$: Total wealth (money in this case)
+ $C$: Cost of living
+ $H$: Total market value of housegrown food, such as veggies and eggs.
+ $R$: Rent
+ $\Delta N$: Number of extra years


To determine the maximum percentage of wealth that can be allocated to the property investment without reducing your financial runway, we set $\Delta N = 0$, and calculate $p$:

$$
p = \frac{R+H}{C+R}
$$

The above equation indicates that to increase p, we can increase the value of housegrown produce ($H$) and/or lower cost of living ($C$).

Now let's visualize an example:
+ $W=4000$
+ $C=950$
+ $H=150$
+ $R=100$


As shown in Figure 1, when the investment percentage  $p≤0.24$  (meaning the price of the house and renovation is ≤960 ), $\Delta N$ becomes positive. This implies that, given these specific wealth and cost-of-living parameters, purchasing a house allows you to sustain your wealth for a longer period than renting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig_count = 1

# set the style to a dark theme
plt.style.use("dark_background")

# match website background
plt.rcParams["figure.facecolor"] = "#181818"
plt.rcParams["axes.facecolor"] = "#181818"
plt.rcParams["axes.edgecolor"] = "#181818"

# ============================================
# Parameters (your example values)
# ============================================
W = 4000      # Total wealth ($)
C = 950       # Annual cost of living ($)
H = 150       # Market value of homegrown food ($/year)
R = 100       # Annual rent ($)

# ============================================
# Define the function for Delta N
# ============================================
def delta_N(p, W, C, H, R):
    """
    p: fraction of wealth spent on house + renovations (0 to 1)
    """
    term1 = (1 - p) * W / (C - H)   # Years if buying
    term2 = W / (C + R)              # Years if renting
    return term1 - term2

# ============================================
# Generate data
# ============================================
p_values = np.linspace(0, 1, 1000)   # p from 0% to 100%
delta_values = delta_N(p_values, W, C, H, R)

# Find where delta_N crosses zero (for vertical line)
# Find the index where delta_N changes sign
idx = np.where(np.diff(np.sign(delta_values)))[0]
if len(idx) > 0:
    p_zero = p_values[idx[0]]  # Approximate p where delta_N = 0
    # Linear interpolation for more precise crossing
    p1, p2 = p_values[idx[0]], p_values[idx[0]+1]
    d1, d2 = delta_values[idx[0]], delta_values[idx[0]+1]
    p_zero_interp = p1 - d1 * (p2 - p1) / (d2 - d1)
else:
    p_zero_interp = None

# ============================================
# Create the plot
# ============================================
fig, ax = plt.subplots(figsize=(8, 6))

# Main curve
ax.plot(p_values, delta_values, linewidth=2.5, color='royalblue', label=r'$\Delta N(p)$')

# Vertical line at delta_N = 0
if p_zero_interp is not None:
    ax.axvline(x=p_zero_interp, color='red', linestyle='--', linewidth=2,
               label=f'$\Delta N = 0$ at $p = {p_zero_interp:.2f}$')
    # Add a horizontal line at y=0 for reference
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)

# Fill regions
ax.fill_between(p_values, 0, delta_values, where=(delta_values > 0), 
                color='pink', alpha=0.25, label='Buying better ($\Delta N > 0$)')
ax.fill_between(p_values, 0, delta_values, where=(delta_values < 0), 
                color='yellow', alpha=0.15, label='Renting better ($\Delta N < 0$)')

# ============================================
# Formatting
# ============================================
ax.set_xlabel('Fraction of wealth spent on house ($p$)', fontsize=12)
ax.set_ylabel('Extra gap years ($\Delta N$)', fontsize=12)

ax.grid(True, alpha=0.3, linestyle=':')
ax.legend(loc='right', fontsize=10)

# Set y-axis limits with some padding
ymax = max(delta_values) * 1.15
ymin = min(delta_values) * 1.15
ax.set_ylim(ymin, ymax)

# Add annotation for key values
if p_zero_interp is not None:
    ax.annotate(f'Break-even\n$p = {p_zero_interp:.2f}$', 
                xy=(p_zero_interp, 0), 
                xytext=(p_zero_interp + 0.1, ymax * 0.3),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                fontsize=11, fontweight='bold', color='red', ha='center')

# Add text box with equation and parameters
textstr = f'$W = {W}$ \n$C = {C}$ \n$H = {H}$ \n$R = {R}$'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.15, 0.72, textstr, transform=ax.transAxes, fontsize=10, 
        color='darkred', verticalalignment='top', bbox=props)

plt.suptitle(
    f"Figure {fig_count}. Renter Optimal Property Investment", y=0.0001, fontsize=12
)
fig_count += 1
plt.tight_layout()
plt.show()



## What happens when cash runs out ##

When the cash is used up, for the two options, there will be two different outcomes:

1. Option 1 (Rent)

    After $N_{\text rent}$ years:
    
    + Cash: $0
    
    + Monthly costs: $C+R$
    
    + Net worth: $0
    
    + Financial pressure: Income to cover monthly costs ($C+R$)

2. Option 2 (Purchase)

    After $N_{\text purchase}$ years:
    
    + Cash: $0
    
    + Monthly costs: $C_o−H$
    
    + Net worth: $C_{\text house}$ (you can sell the house)
    
    + Financial pressure: Lower — you can:
    
        + Sell the house for cash
    
        + Earn income to cover **$Co−H$** 
 
Due to that there is still net worth left for Option 2, that leads to another option: selling the house to get more gap time.

## Advanced Calculation of $\Delta N$ ##

When the cash runs out, Option 2 leaves us with residual net worth (the house), it unlocks a third pathway not available to the renter: selling the property to regain liquidity. This action extends your financial runway, creating the "gap time" - $N_{\text sell}$, which will be calculated referencing the [article](https://diogenesanalytics.com/blog/2024/05/12/rent-or-buy) from Diogenes Analytics. This third option extends the second: it represents the additional years gained after selling the property purchased in the second option.

$$
\Delta N = N_{\text purchase} - N_{\text rent} + N_{\text sell} - N_{\text limit}
$$

Where:

+ $N_{\text limit}$: a threshold value representing the minimum number of years one must hold the property before selling to make it worthwhile. This term serves primarily to identify the advantage threshold — i.e. the break-even point that delineates where the advantage of purchasing begins to erode.


$$
N_{\text sell} = a \cdot N_{\text purchase}
$$

Where:
+ $a$: the appreciation (as a fraction) of the property.

That leads us to:

$$
\Delta N = \frac{(1-p)W}{(C-H)} - \frac{W}{(C+R)} + a \cdot \frac{pW}{C+R} - N_{\text limit}
$$

$$
\Delta N = \frac{(1-p)W}{(C-H)} - \frac{W-apW}{(C+R)} - N_{\text limit}
$$

Now let us apply these formulas to the previous example ($W=4000, C=950, H=150, R=100$), setting $N_{\text limit}=1$ and $a=1$ (i.e., the property price neither appreciates nor depreciates).

As shown in Figure 2, when $p$ ≤ 0.16 (meaning the purchase and renovation cost of the house is ≤ 640 ), Option 3 provides more than one additional year compared to Option 1 (renting an apartment).

Although comparing renting versus purchasing through the time dimension is straightforward, these two choices have broader implications for our **overall position** — encompassing factors such as health, skills, and overall wealth. In the next section, we will compare these two options using the **utility function of position**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================
# Parameters
# ============================================
W = 4000       # Total wealth
C = 950        # Cost of living
H = 150        # Homegrown food value
R = 100        # Rent
a = 1.0        # Appreciation fraction (1 = no change)
N_limit = 1  # Minimum limit of months worth selling for

# ============================================
# Define Delta N for Option 3
# ============================================
def delta_N_option3(p, W, C, H, R, a, N_limit):
    """
    Delta N = (1-p)W/(C-H) - (W - a*p*W)/(C+R) - N_limit
    """
    term1 = (1 - p) * W / (C - H)           # Purchase years
    term2 = (W - a * p * W) / (C + R)        # Rent years adjusted for selling
    return term1 - term2 - N_limit

# ============================================
# Generate data
# ============================================
p_values = np.linspace(0, 1, 1000)
delta_values = delta_N_option3(p_values, W, C, H, R, a, N_limit)

# Find break-even point (where delta_N crosses zero)
idx = np.where(np.diff(np.sign(delta_values)))[0]
if len(idx) > 0:
    p1, p2 = p_values[idx[0]], p_values[idx[0]+1]
    d1, d2 = delta_values[idx[0]], delta_values[idx[0]+1]
    p_zero = p1 - d1 * (p2 - p1) / (d2 - d1)
else:
    p_zero = None

# ============================================
# Create the plot
# ============================================
fig, ax = plt.subplots(figsize=(11, 7))

# Main curve
ax.plot(p_values, delta_values, linewidth=2.5, color='royalblue',
        label=r'$\Delta N(p)$ (Option 3)')

# Vertical line at delta_N = 0
if p_zero is not None:
    ax.axvline(x=p_zero, color='red', linestyle='--', linewidth=2,
               label=f'Break-even at $p = {p_zero:.3f}$')

# Horizontal line at y=0
ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.4)

# Shaded regions
ax.fill_between(p_values, 0, delta_values, where=(delta_values > 0),
                color='pink', alpha=0.25, label='Option 3 better ($\Delta N > 0$)')
ax.fill_between(p_values, 0, delta_values, where=(delta_values < 0),
                color='yellow', alpha=0.15, label='Option 1 better ($\Delta N < 0$)')

# ============================================
# Formatting
# ============================================
ax.set_xlabel('Fraction of wealth spent on house ($p$)', fontsize=13)
ax.set_ylabel('Extra gap years ($\Delta N$)', fontsize=13)


ax.grid(True, alpha=0.3, linestyle=':')
ax.legend(loc='best', fontsize=11)

# Annotation
if p_zero is not None:
    ymax = max(delta_values) * 1.15
    ax.annotate(f'Break-even\n$p = {p_zero:.2f}$\n(House = ${p_zero*W:.0f})',
                xy=(p_zero, 0),
                xytext=(p_zero + 0.12, ymax * 0.35),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                fontsize=11, fontweight='bold', color='red', ha='center',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                          edgecolor='red', alpha=0.9))



plt.suptitle(
    f"Figure {fig_count}. Option 3: Purchase, Sell, Rent", y=0.0001, fontsize=12
)
fig_count += 1

plt.tight_layout()
plt.show()

## Define the Game ##

While $\Delta N$ captures the time dimencion of the options, a comprehensive comparison requires a broader perspective. Following the [framework of decision-making](https://diogenesanalytics.com/blog/2026/03/05/universal-decision-making-framework)  from Diogenes Analytics, we evaluate actions not by their immediate payoff, but by how they reshape the structure of reachable futures.

To formalize this comparison, we first define the game in order to establish what constitutes success or failure.

Following the framework, a game is defined as:

$$
G = (S, A, f, S_T, V)
$$

where:

+ $S$ = set of all admissible system states
+ $A(s)$ = set of legal actions available at state
+ $f$: $S \times A \to S \quad \text{(transition function)} $
+ $S_T$ $\subseteq S \quad \text{(terminal states)}$ 
+ $V : S_T \to \mathbb{R} \quad \text{(terminal value function)}$

In our game, the actions are:

$$
A(s) = \left\{ A_{\text{rent}}, A_{\text{purchase}} \right\}
$$

Each action transforms the current state into different terminal states:

$$
S'_{\text rent} = f(S, A_{rent})
$$

$$
S'_{\text purchase} = f(S, A_{purchase})
$$

The rational choice is the action that maximizes the utility gain:

$$
A^* = \arg\max_{A \in \{A_{\text{rent}}, A_{\text{purchase}}\}} \Delta U(P)
$$

Therefore:

$$
\text{Choose purchase} \iff U(P'_{\text{purchase}}) > U(P'_{\text{rent}})
$$

$$
\text{Choose rent} \iff U(P'_{\text{rent}}) > U(P'_{\text{purchase}})
$$

$$
\text{Indifferent} \iff U(P'_{\text{purchase}}) = U(P'_{\text{rent}})
$$

Now let's calculate the position utility for each action.

## Position Utility ##

In reference to the [utility equation of position](https://diogenesanalytics.com/blog/2026/03/05/universal-decision-making-framework) develped by Diogenes Analytics, to evaluate the quality of a position, the framework decomposes utility into four structural components: advantage, threat, success diversity, and failure diversity. Together, these yield the unified position utility:

$$
U(P) = A(P) - T(P) + D^{+}(P) - D^{-}(P).
$$

Where:
+ access to positive terminal value $A(P)$
+ exposure to negative terminal value $T(P)$
+ structural richness of success paths $D^{+}(P)$
+ structural fragility of failure paths $D^{-}(P)$

## Define Terminal State ##

In order to calculate the utility, first we need to define our terminal state ($S_T$) based on our goals:

$$
S_T = (W, F, S, H, N)
$$

Where:
+ $W$: Wealth (residual assets when cash runs out)
+ $F$: Number of food sources
+ $S$: Skills (such as property transaction, DIY skills)
+ $N$: Environmental noise level 

Note that $W$, $F$, and $S$ are benefits (higher is better), while $N$ is a cost (lower is better). This distinction matters when we assign values.

Next, we define the terminal value function $V$, which scores each terminal state $o$:

$$
V(o) = w_W \cdot W(o) + w_F \cdot F(o) + w_S \cdot S(o) - w_N \cdot N(o)
$$

Where $w$s reflect our personal priorities. In our folliwng calculation, we assign equal weights to each factor:

$$
w_W = 0.25, w_F  = 0.25, w_S = 0.25, w_N = 0.25
$$

As for the **current position $P$**, we set each criterion to zero:
$$P=(W,F,S,N)=(0,0,0,0)$$

This establishes a neutral baseline: any positive value represents an improvement over the status quo, and any negative value represents a deterioration.

## Calculation of Each Term ##

Here's how to instantiate each term for our rent vs. purchase problem:

+ Advantage $A(P)$

    The sum of positive terminal values reachable from your position.

  + For purchase:

    + Grow food and sell house at break-even

    + Grow food, develop other revenues, keep house forever
   
    + Grow food and sell house at profit

    For all positive outcomes:
   
      $$A(P_{\text purchase}) = \sum V_p(o)$$ 
  + For rent:
    
    + Develop new revenue streams while living the in the same apartment
      
    For all positive outcomes:
      $$A(P_{\text rent}) = \sum V_r(o)$$

The calculation of $A(P)$ is shown in Tables 1 and 2. $A(p_{\text purchase})$ is 22, which is 6.8 times of $A_{\text rent}$ (3.25). This gap reflects two compounding effects.

First, purchase has more positive terminal states. Rent reaches only one positive state ("stay the same"), while purchase reaches three distinct states (sell at break-even, develop revenue and keep the house, sell at profit), more possibilities for the future.

Second, each of purchase's positive states is worth more than rent's entire advantage. The lowest-valued purchase state exceeds $A_{\text rent}$ on its own.

Together, these two effects mean $A(P)$ captures something richer than simply "how good" a position is — **it captures how many ways you can be good, and how much each of those ways is worth.** This is the "structural richness" the utility function is designed to measure.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig_count = 1

# Set the style to a dark theme
plt.style.use("dark_background")

# Match website background
plt.rcParams["figure.facecolor"] = "#181818"
plt.rcParams["axes.facecolor"] = "#181818"
plt.rcParams["axes.edgecolor"] = "#181818"


# ============================================
# WEIGHTS (must sum to 1)
# ============================================
w_W = 0.25   # Wealth
w_F = 0.25   # Food sources
w_S = 0.25   # Skills
w_N = 0.25   # Quietness

weights = [w_W, w_F, w_S, w_N]

# ============================================
# COMPUTE V(o) FOR EACH OUTCOME
# ============================================
def compute_V(row, weights):
    """V(o) = w_W*W + w_F*F + w_S*S + w_N*N"""
    scores = row[1:]  # [W, F, S, N]
    return sum(w * s for w, s in zip(weights, scores))

# ============================================
# DATA: Purchase — Positive Outcomes
# ============================================
purchase_outcomes = [
    ["Grow food, sell at break-even", 0, 1, 1, -1],
    ["Develop new revenue streams, \nkeep house forever", 1, 1, 1, -1],
    ["Grow food, sell at profit",  1, 1, 1, -1],
]

# ============================================
# DATA: Rent — Positive Outcomes
# ============================================
rent_outcomes = [
    ["Develop new revenue streams \nwhile living the in the apartment", 1, 0, 0, 0],
]

# ============================================
# BUILD TABLE DATA
# ============================================
def build_table_data(outcomes, weights):
    table_data = []
    for row in outcomes:
        V = compute_V(row, weights)
        table_data.append(row + [round(V, 2)])
    return table_data

purchase_data = build_table_data(purchase_outcomes, weights)
rent_data = build_table_data(rent_outcomes, weights)

# ============================================
# COMPUTE A(P) FOR EACH OPTION
# ============================================
A_purchase = sum(row[-1] for row in purchase_data)
A_rent = sum(row[-1] for row in rent_data)

# ============================================
# COLUMN HEADERS
# ============================================
col_labels = [
    "Terminal State",
    "$W$\n(Wealth)",
    "$F$\n(Food)",
    "$S$\n(Skills)",
    "$N$\n(Quiet)",
    "$V(o)$\n(Value)",
]

# ============================================
# CREATE FIGURE
# ============================================
fig = plt.figure(figsize=(8, 6))

# ============================================
# AXES 1 (TOP): PURCHASE TABLE
# ============================================
ax1 = fig.add_axes([0.05, 0.50, 0.90, 0.32])
ax1.axis('off')

table_purchase = ax1.table(
    cellText=purchase_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.32, 0.10, 0.10, 0.10, 0.10, 0.14],
)

table_purchase.auto_set_font_size(False)
table_purchase.set_fontsize(11)
table_purchase.scale(1.2, 2.0)

# Style purchase table
for j in range(len(col_labels)):
    cell = table_purchase[(0, j)]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)

for i in range(1, len(purchase_data) + 1):
    for j in range(len(col_labels)):
        cell = table_purchase[(i, j)]
        if i % 2 == 0:
            cell.set_facecolor('#F4F6F7')
        else:
            cell.set_facecolor('#FFFFFF')
        cell.set_edgecolor('#BDC3C7')
        cell.set_linewidth(0.8)
        cell.set_text_props(color='black')
        if j == len(col_labels) - 1:
            cell.set_facecolor('#E8F8F5')
            cell.set_text_props(fontweight='bold', color='#1E8449')

# Purchase title
ax1.set_title(
    "Table 1. Purchase Option — Positive Terminal States & Value Calculation",
    fontsize=13, fontweight='bold', pad=5, color='white'
)

# ============================================
# PURCHASE A(P) FOOTER
# ============================================
ax1.text(
    0.5, -0.08,
    rf"$A(P_{{\text{{purchase}}}}) = \sum V(o) = {A_purchase:.2f}$",
    transform=ax1.transAxes,
    fontsize=13, fontweight='bold', color='#1E8449',
    ha='center', va='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#FADBD8',
              edgecolor='#1E8449', linewidth=1.5)
)

# ============================================
# AXES 2 (BOTTOM): RENT TABLE
# ============================================
ax2 = fig.add_axes([0.05, 0.10, 0.90, 0.20])
ax2.axis('off')

table_rent = ax2.table(
    cellText=rent_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.32, 0.10, 0.10, 0.10, 0.10, 0.14],
)

table_rent.auto_set_font_size(False)
table_rent.set_fontsize(11)
table_rent.scale(1.2, 2.2)

# Style rent table
for j in range(len(col_labels)):
    cell = table_rent[(0, j)]
    cell.set_facecolor('#922B21')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)

for i in range(1, len(rent_data) + 1):
    for j in range(len(col_labels)):
        cell = table_rent[(i, j)]
        cell.set_facecolor('#FFFFFF')
        cell.set_edgecolor('#BDC3C7')
        cell.set_linewidth(0.8)
        cell.set_text_props(color='black')
        if j == len(col_labels) - 1:
            cell.set_facecolor('#FADBD8')
            cell.set_text_props(color='black', fontweight='bold')

# Rent title
ax2.set_title(
    "Table 2. Rent Option — Positive Terminal States & Value Calculation",
    fontsize=13, fontweight='bold', pad=5, color='white'
)

# ============================================
# RENT A(P) FOOTER
# ============================================
ax2.text(
    0.5, -0.18,
    rf"$A(P_{{\text{{rent}}}}) = \sum V(o) = {A_rent:.2f}$",
    transform=ax2.transAxes,
    fontsize=13, fontweight='bold', color='#E74C3C',
    ha='center', va='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#FADBD8', 
              edgecolor='#E74C3C', linewidth=1.5)
)

plt.show()

+ Threat $T(P)$

    The sum of negative terminal values.

  + For purchase:

    + Risk of selling house at a lower than purchased price
    
    + Risk of maintenance costs exceeding budget
    
    + Risk of food production not reaching expectation

    over all negative outcomes:
   
      $T(P_{\text purchase}) = \sum |V(o)|$
   
  + For rent:
 
    + Being forced to move out and rent another apartment

    over all negative outcomes:
   
      $T(P_{\text rent}) = \sum |V(o)|$
 

The calculation of $T(P)$ is shown in Tables 3 and 4. $T(P_{\text purchase})$ is 16.25, which is 3.25 times $T(P_{\text rent})$ (5). This gap reflects two compounding effects.

First, purchase has more negative terminal states. Rent is exposed to two failure modes (eviction and rent rising), while purchase is exposed to three (selling below purchase price, maintenance exceeding budget, and food production failing).

However, each of purchase's negative states is still worth more than rent's (5.25 vs. 2.5).

Together, these two effects reveal something important: even though purchase has more ways for things to go wrong, the worst-case outcome in each of those failure states is still better than what renting offers in its failure states.



In [ ]:
# Data for T(P)

# ============================================
# WEIGHTS (must sum to 1)
# ============================================
w_W = 0.25   # Wealth
w_F = 0.25   # Food sources
w_S = 0.25   # Skills
w_N = 0.25   # Quietness

weights = [w_W, w_F, w_S, w_N]

# ============================================
# COMPUTE |V(o)| FOR NEGATIVE OUTCOMES
# ============================================
def compute_absV(row, weights):
    """|V(o)| = | w_W*W + w_F*F + w_S*S + w_N*N | for negative outcomes"""
    scores = row[1:]  # [W, F, S, N]
    raw_V = sum(w * s for w, s in zip(weights, scores))
    return raw_V # Take absolute value

# ============================================
# DATA: Purchase — Negative Outcomes (Threats)
# ============================================
purchase_negative = [
    # [Terminal State, W, F, S, N]
    # Lower scores = worse outcome
    ["Lose money while holding the property",  -1, 1, 1, -1],
    ["Food production fails",                  -1, 0, 1, -1],
]

# ============================================
# DATA: Rent — Negative Outcomes (Threats)
# ============================================
rent_negative = [
    # [Terminal State, W, F, S, N]
    ["Forced move",        -1, 0, 0, -1],
]

# ============================================
# BUILD TABLE DATA
# ============================================
def build_threat_data(outcomes, weights):
    table_data = []
    for row in outcomes:
        absV = compute_absV(row, weights)
        table_data.append(row + [round(absV, 2)])
    return table_data

purchase_threat_data = build_threat_data(purchase_negative, weights)
rent_threat_data = build_threat_data(rent_negative, weights)

# Compute T(P)
T_purchase = sum(row[-1] for row in purchase_threat_data)
T_rent = sum(row[-1] for row in rent_threat_data)

# ============================================
# COLUMN HEADERS
# ============================================
col_labels = [
    "Terminal State",
    "$W$\n(Wealth)",
    "$F$\n(Food)",
    "$S$\n(Skills)",
    "$N$\n(Quiet)",
    "$V(o)$",
]

# ============================================
# CREATE FIGURE
# ============================================
fig = plt.figure(figsize=(10, 6))

# ============================================
# AXES 1 (TOP): PURCHASE THREATS TABLE
# ============================================
ax1 = fig.add_axes([0.05, 0.55, 0.90, 0.30])
ax1.axis('off')

table_purchase = ax1.table(
    cellText=purchase_threat_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.34, 0.10, 0.10, 0.10, 0.10, 0.14],
)

table_purchase.auto_set_font_size(False)
table_purchase.set_fontsize(11)
table_purchase.scale(1.2, 2.0)

# Style purchase table
for j in range(len(col_labels)):
    cell = table_purchase[(0, j)]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)

# FIX: Used purchase_threat_data instead of purchase_data
for i in range(1, len(purchase_threat_data) + 1):
    for j in range(len(col_labels)):
        cell = table_purchase[(i, j)]
        if i % 2 == 0:
            cell.set_facecolor('#F4F6F7')
        else:
            cell.set_facecolor('#FFFFFF')
        cell.set_edgecolor('#BDC3C7')
        cell.set_linewidth(0.8)
        cell.set_text_props(color='black')
        if j == len(col_labels) - 1:
            cell.set_facecolor('#E8F8F5')
            cell.set_text_props(fontweight='bold', color='#1E8449')

# Purchase threats title
ax1.set_title(
    "Table 3. Purchase Option — Negative Terminal States (Threats)",
    fontsize=13, fontweight='bold', pad=15, color='white'
)

# Purchase T(P) footer
ax1.text(
    0.5, -0.12,
    rf"$T(P_{{\text{{purchase}}}}) = \sum |V(o)| = {T_purchase:.2f}$",
    transform=ax1.transAxes,
    fontsize=13, fontweight='bold', color='#1E8449',
    ha='center', va='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#FADBD8',
              edgecolor='#1E8449', linewidth=1.5)
)

# ============================================
# AXES 2 (BOTTOM): RENT THREATS TABLE
# ============================================
ax2 = fig.add_axes([0.05, 0.15, 0.90, 0.20])
ax2.axis('off')

table_rent = ax2.table(
    cellText=rent_threat_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.36, 0.10, 0.10, 0.10, 0.10, 0.14],
)

table_rent.auto_set_font_size(False)
table_rent.set_fontsize(11)
table_rent.scale(1.2, 2.2)

# Style rent threats table
for j in range(len(col_labels)):
    cell = table_rent[(0, j)]
    cell.set_facecolor('#922B21')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)

for i in range(1, len(rent_threat_data) + 1):
    for j in range(len(col_labels)):
        cell = table_rent[(i, j)]
        cell.set_facecolor('#FFFFFF')
        cell.set_edgecolor('#BDC3C7')
        cell.set_linewidth(0.8)
        cell.set_text_props(color='black')
        if j == len(col_labels) - 1:
            cell.set_facecolor('#FADBD8')
            cell.set_text_props(color='black', fontweight='bold')

# Rent threats title
ax2.set_title(
    "Table 4. Rent Option — Negative Terminal States (Threats)",
    fontsize=13, fontweight='bold', pad=15, color='white'
)

# Rent T(P) footer
ax2.text(
    0.5, -0.25,
    rf"$T(P_{{\text{{rent}}}}) = \sum |V(o)| = {T_rent:.2f}$",
    transform=ax2.transAxes,
    fontsize=13, fontweight='bold', color='black',
    ha='center', va='top',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FADBD8',
              edgecolor='#922B21', linewidth=1.5)
)

plt.show()

+ Success Diversity $D^+(P)$

    $D^+(P)$ is the log of the number of distinct paths to success.

    The calculation of $D^+(P)$ is shown in Tables 5 and 6. $D^+(P_{\text purchase})$ is 1.39, which is twice of$D^+(P_{\text rent})$  (0.69). This gap shows that purchase has more distinct paths to success than rent (4 vs. 2). With more ways to succeed, purchase offers a higher structural probability of reaching a positive outcome than rent.

In [ ]:
import math

## plot the D+(P) tables

# ============================================
# DATA: 4 Purchase Paths
# ============================================
purchase_paths = [
    ["Path 1: Buy → develop revenue\nand farm → keep house",       "↑↑", "↑↑", "↑↑", "↑↑"],
    ["Path 2: Buy → don't farm →\ndevelop revenue → keep house",   "↑↑", "→",  "↑",  "↑↑"],
    ["Path 3: Buy → farm →\nsell at profit",                       "↑↑", "↑↑", "↑↑", "↑"],
    ["Path 4: Buy → farm →\nsell at break-even",                   "↑",  "↑↑", "↑↑", "↑"],
]

# ============================================
# DATA: 2 Rent Paths
# ============================================
rent_paths = [
    ["Path 1: Find other apts nearby with \nthe same rent",  "→",  "→",  "→",  "→"],
    ["Path 2: Stay in the same apartment",      "→",  "→",  "→",  "→"],
]

# ============================================
# COMPUTE D+(P)
# ============================================
D_plus_purchase = math.log(len(purchase_paths))
D_plus_rent = math.log(len(rent_paths))

# ============================================
# COLUMN HEADERS
# ============================================
col_labels = [
    "Path",
    "$W$\n(Wealth)",
    "$F$\n(Food)",
    "$S$\n(Skills)",
    "$N$\n(Quiet)",
]

# ============================================
# CREATE FIGURE
# ============================================
fig = plt.figure(figsize=(7, 7))
fig.patch.set_facecolor('#181818')

# ============================================
# AXES 1 (TOP): PURCHASE TABLE
# ============================================
ax1 = fig.add_axes([0.05, 0.52, 0.90, 0.40])
ax1.axis('off')

table_p = ax1.table(
    cellText=purchase_paths,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.42, 0.145, 0.145, 0.145, 0.145],
)
table_p.auto_set_font_size(False)
table_p.set_fontsize(10)
table_p.scale(1.2, 2.2)

# Style purchase header
for j in range(len(col_labels)):
    cell = table_p[(0, j)]
    cell.set_facecolor('#1E8449')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)

# Style purchase body
for i in range(1, len(purchase_paths) + 1):
    for j in range(len(col_labels)):
        cell = table_p[(i, j)]
        cell.set_facecolor('#1F2A36' if i % 2 == 0 else '#2A3744')
        cell.set_edgecolor('#4A5A6A')
        cell.set_linewidth(0.8)
        if j == 0:
            cell.set_text_props(color='white', fontsize=9, ha='left')
        else:
            text = purchase_paths[i-1][j]
            if "↑↑" in text:
                cell.set_text_props(color='#2ECC71', fontweight='bold', fontsize=12)
            elif "↑" in text:
                cell.set_text_props(color='#7DCEA0', fontweight='bold', fontsize=12)
            elif "↓" in text:
                cell.set_text_props(color='#E74C3C', fontweight='bold', fontsize=12)
            elif "↓↓" in text:
                cell.set_text_props(color='#C0392B', fontweight='bold', fontsize=12)
            else:
                cell.set_text_props(color='#BDC3C7', fontweight='bold', fontsize=12)

# Purchase title
ax1.set_title(
    "Table 5. Purchase Option — Paths and Influence on W, F, S, N",
    fontsize=13, fontweight='bold', pad=10, color='white'
)

# ============================================
# PURCHASE D+(P) FOOTER (inside the plot area, below the table)
# ============================================
ax1.text(
    0.5, 0.04,
    rf"$D^+(P_{{\text{{purchase}}}}) = \log(|\Pi^+|) = "
    rf"\log({len(purchase_paths)}) = {D_plus_purchase:.2f}$",
    transform=ax1.transAxes,
    fontsize=12, fontweight='bold', color='white',
    ha='center', va='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#0E2A1E',
              edgecolor='#2ECC71', linewidth=1.5)
)

# ============================================
# AXES 2 (BOTTOM): RENT TABLE
# ============================================
ax2 = fig.add_axes([0.05, 0.15, 0.90, 0.22])
ax2.axis('off')

table_r = ax2.table(
    cellText=rent_paths,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.42, 0.145, 0.145, 0.145, 0.145],
)
table_r.auto_set_font_size(False)
table_r.set_fontsize(10)
table_r.scale(1.2, 2.4)

# Style rent header
for j in range(len(col_labels)):
    cell = table_r[(0, j)]
    cell.set_facecolor('#922B21')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)

# Style rent body
for i in range(1, len(rent_paths) + 1):
    for j in range(len(col_labels)):
        cell = table_r[(i, j)]
        cell.set_facecolor('#2A1F1F' if i % 2 == 0 else '#3A2525')
        cell.set_edgecolor('#5A3A3A')
        cell.set_linewidth(0.8)
        if j == 0:
            cell.set_text_props(color='white', fontsize=10, ha='left')
        else:
            text = rent_paths[i-1][j]
            if "↑↑" in text:
                cell.set_text_props(color='#2ECC71', fontweight='bold', fontsize=12)
            elif "↑" in text:
                cell.set_text_props(color='#7DCEA0', fontweight='bold', fontsize=12)
            elif "↓" in text:
                cell.set_text_props(color='#E74C3C', fontweight='bold', fontsize=12)
            elif "↓↓" in text:
                cell.set_text_props(color='#C0392B', fontweight='bold', fontsize=12)
            else:
                cell.set_text_props(color='#BDC3C7', fontweight='bold', fontsize=12)

# Rent title
ax2.set_title(
    "Table 6. Rent Option — Paths and Influence on W, F, S, N",
    fontsize=13, fontweight='bold', pad=12, color='white'
)

# ============================================
# RENT D+(P) FOOTER (inside the plot area, below the table)
# ============================================
ax2.text(
    0.5, -0.18,
    rf"$D^+(P_{{\text{{rent}}}}) = \log(|\Pi^+|) = "
    rf"\log({len(rent_paths)}) = {D_plus_rent:.2f}$",
    transform=ax2.transAxes,
    fontsize=12, fontweight='bold', color='white',
    ha='center', va='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#2A0E0E',
              edgecolor='#E74C3C', linewidth=1.5)
)

# ============================================
# LEGEND (bottom of figure)
# ============================================
legend_text = (
    r"$\uparrow\uparrow$ = strong increase    "
    r"$\uparrow$ = increase    "
    r"$\rightarrow$ = neutral    "
    r"$\downarrow$ = decrease    "
    r"$\downarrow\downarrow$ = strong decrease"
)
fig.text(
    0.5, 0.02, legend_text,
    fontsize=11, color='white',
    ha='center', va='bottom',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#2C3E50',
              edgecolor='white', linewidth=1.5)
)

plt.show()


+ Failure Diversity $D^−(P)$

    $D^−(P)$ is the log of distinct paths to failure.

    Table 7 and 8 show the calculation of $D^-(P)$. $D^−(P_{\text purchase})$ is slightly higher than $D^−(P_{\text rent})$ (1.39 vs. 1.10) because purchase has four ways to failure white rent has three.  

In [ ]:
# Plot D-(P) tables

# ============================================
# DATA: 4 Purchase Failure Paths
# ============================================
purchase_failure_paths = [
    ["Path 1: House needs major repair\n→ lose money",              "↓", "↑",  "↑",  "↑"],
    ["Path 2: Cannot sell at even price → lose money",              "↓", "↑",  "↑",  "↑"],
    ["Path 3: Crop failure → Yield is \nlower than expected",          "↓",  "→", "↑",  "↑"],
    ["Path 4: Unexpected noise in the\nneighborhood → environment worsens", "→", "↑", "↑", "↓"],
]

# ============================================
# DATA: 2 Rent Failure Paths
# ============================================
rent_failure_paths = [
    ["Path 1: Rent increases → move",     "↓",  "→",  "→",  "→"],
    ["Path 2: Landlord evicts → move",    "↓", "→",  "→",  "↓"],
    ["Path 3: Noise increases from \nneighbors → move", "↓", "→",  "→",  "↓"],
]

# ============================================
# COMPUTE D-(P)
# ============================================
D_minus_purchase = math.log(len(purchase_failure_paths))
D_minus_rent = math.log(len(rent_failure_paths))

# ============================================
# COLUMN HEADERS
# ============================================
col_labels = [
    "Failure Path",
    "$W$\n(Wealth)",
    "$F$\n(Food)",
    "$S$\n(Skills)",
    "$N$\n(Quiet)",
]

# ============================================
# CREATE FIGURE
# ============================================
fig = plt.figure(figsize=(7, 7))
fig.patch.set_facecolor('#181818')

# ============================================
# AXES 1 (TOP): PURCHASE FAILURE TABLE
# ============================================
ax1 = fig.add_axes([0.05, 0.52, 0.90, 0.40])
ax1.axis('off')

table_p = ax1.table(
    cellText=purchase_failure_paths,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.42, 0.145, 0.145, 0.145, 0.145],
)
table_p.auto_set_font_size(False)
table_p.set_fontsize(10)
table_p.scale(1.2, 2.2)

# Style purchase header
for j in range(len(col_labels)):
    cell = table_p[(0, j)]
    cell.set_facecolor('#1E8449')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)

# Style purchase body
for i in range(1, len(purchase_paths) + 1):
    for j in range(len(col_labels)):
        cell = table_p[(i, j)]
        cell.set_facecolor('#1F2A36' if i % 2 == 0 else '#2A3744')
        cell.set_edgecolor('#4A5A6A')
        cell.set_linewidth(0.8)
        if j == 0:
            cell.set_text_props(color='white', fontsize=9, ha='left')
        else:
            text = purchase_paths[i-1][j]
            if "↑↑" in text:
                cell.set_text_props(color='#2ECC71', fontweight='bold', fontsize=12)
            elif "↑" in text:
                cell.set_text_props(color='#7DCEA0', fontweight='bold', fontsize=12)
            elif "↓" in text:
                cell.set_text_props(color='#E74C3C', fontweight='bold', fontsize=12)
            elif "↓↓" in text:
                cell.set_text_props(color='#C0392B', fontweight='bold', fontsize=12)
            else:
                cell.set_text_props(color='#BDC3C7', fontweight='bold', fontsize=12)

# Purchase title
ax1.set_title(
    "Table 7. Purchase Option — Failure Paths and Influence on W, F, S, N",
    fontsize=13, fontweight='bold', pad=12, color='white'
)

# ============================================
# PURCHASE D-(P) FOOTER
# ============================================
ax1.text(
    0.5, -0.01,
    rf"$D^-(P_{{\text{{purchase}}}}) = \log(|\Pi^-|) = "
    rf"\log({len(purchase_failure_paths)}) = {D_minus_purchase:.2f}$",
    transform=ax1.transAxes,
    fontsize=12, fontweight='bold', color='white',
    ha='center', va='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#2C3E50',
              edgecolor='#2ECC71', linewidth=1.5)
)



# ============================================
# AXES 2 (BOTTOM): RENT FAILURE TABLE
# ============================================
ax2 = fig.add_axes([0.05, 0.15, 0.90, 0.22])
ax2.axis('off')

table_r = ax2.table(
    cellText=rent_failure_paths,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.42, 0.145, 0.145, 0.145, 0.145],
)
table_r.auto_set_font_size(False)
table_r.set_fontsize(10)
table_r.scale(1.2, 2.4)

# Style rent header
for j in range(len(col_labels)):
    cell = table_r[(0, j)]
    cell.set_facecolor('#922B21')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)

# Style rent body
for i in range(1, len(rent_failure_paths) + 1):
    for j in range(len(col_labels)):
        cell = table_r[(i, j)]
        cell.set_facecolor('#2A1F1F' if i % 2 == 0 else '#3A2525')
        cell.set_edgecolor('#5A3A3A')
        cell.set_linewidth(0.8)
        if j == 0:
            cell.set_text_props(color='white', fontsize=10, ha='left')
        else:
            text = rent_failure_paths[i-1][j]
            if "↓↓" in text:
                cell.set_text_props(color='#C0392B', fontweight='bold', fontsize=12)
            elif "↓" in text:
                cell.set_text_props(color='#E74C3C', fontweight='bold', fontsize=12)
            elif "↑↑" in text:
                cell.set_text_props(color='#2ECC71', fontweight='bold', fontsize=12)
            elif "↑" in text:
                cell.set_text_props(color='#7DCEA0', fontweight='bold', fontsize=12)
            else:
                cell.set_text_props(color='#BDC3C7', fontweight='bold', fontsize=12)

# Rent title
ax2.set_title(
    "Talbe 8. Rent Option — Failure Paths and Influence on W, F, S, N",
    fontsize=13, fontweight='bold', pad=12, color='white'
)

# ============================================
# RENT D-(P) FOOTER
# ============================================
ax2.text(
    0.5, -0.25,
    rf"$D^-(P_{{\text{{rent}}}}) = \log(|\Pi^-|) = "
    rf"\log({len(rent_failure_paths)}) = {D_minus_rent:.2f}$",
    transform=ax2.transAxes,
    fontsize=12, fontweight='bold', color='white',
    ha='center', va='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#2C3E50',
              edgecolor='#E74C3C', linewidth=1.5)
)

# ============================================
# LEGEND (bottom)
# ============================================
legend_text = (
    r"$\uparrow\uparrow$ = strong increase    "
    r"$\uparrow$ = increase    "
    r"$\rightarrow$ = neutral    "
    r"$\downarrow$ = decrease    "
    r"$\downarrow\downarrow$ = strong decrease"
)
fig.text(
    0.5, -0.02, legend_text,
    fontsize=11, color='white',
    ha='center', va='bottom',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#2C3E50',
              edgecolor='white', linewidth=1.5)
)


plt.show()



## Calculation of Utility ##

Based on the caculation of each term above, now we can calculate the utility value for each option. As shown in Figure 1, 



## Key Insight from the Article ##

The article's central claim is:

    "Strategic decisions are evaluated not only by immediate effects, but by how they reshape the geometry of future possibility."

The $\Delta N$ equation captures one dimension of this (time). The full utility function captures all dimensions simultaneously:

+ Time (years gained)

+ Assets (what you own at the end)

+ Skills

+ Robustness (path diversity)

This is why the purchase option wins even when $\Delta N$ is small or zero — because it improves the structure of the future, not just the length of it.

In [ ]:

import math

# ============================================
# DATA
# ============================================
# Utility calculation
U_purchase = A_purchase - T_purchase + D_plus_purchase - D_minus_purchase
U_rent = A_rent - T_rent + D_plus_rent - D_minus_rent

# ============================================
# BUILD TABLE DATA
# ============================================
table_data = [
    ["$A(P)$ — Advantage",           f"{A_purchase:.2f}", f"{A_rent:.2f}"],
    ["$T(P)$ — Threat",              f"{T_purchase:.2f}", f"{T_rent:.2f}"],
    ["$D^+(P)$ — Success Diversity", f"{D_plus_purchase:.2f}", f"{D_plus_rent:.2f}"],
    ["$D^-(P)$ — Failure Diversity", f"{D_minus_purchase:.2f}", f"{D_minus_rent:.2f}"],
    ["$U(P)$ — Net Utility",         f"{U_purchase:.2f}", f"{U_rent:.2f}"]
]

col_labels = [
    "Component",
    "Purchase",
    "Rent",
]

# ============================================
# CREATE FIGURE
# ============================================
fig = plt.figure(figsize=(7, 3))
fig.patch.set_facecolor('#181818')

ax = fig.add_axes([0.05, 0.10, 0.90, 0.80])
ax.axis('off')

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.32, 0.18, 0.18],
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.2)

# ============================================
# STYLE HEADER
# ============================================
for j in range(len(col_labels)):
    cell = table[(0, j)]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', fontweight='bold', fontsize=12)
    cell.set_edgecolor('white')
    cell.set_linewidth(1.5)

# ============================================
# STYLE BODY
# ============================================

for i in range(1, len(table_data) + 1):
    for j in range(len(col_labels)):
        cell = table[(i, j)]

        # Last row = U(P) — highlight
        if i == len(table_data):
            if j == 0:
                cell.set_facecolor('#1A5276')
                cell.set_text_props(color='white', fontweight='bold', fontsize=12, ha='left')
            elif j == 1:
                cell.set_facecolor('#0E2A1E')
                cell.set_text_props(color='#E74C3C', fontweight='bold', fontsize=13)
            elif j == 2:
                cell.set_facecolor('#2A0E0E')
                cell.set_text_props(color='#2ECC71', fontweight='bold', fontsize=13)
        else:
            # Alternate row colors
            if i % 2 == 0:
                cell.set_facecolor('#1F2A36')
            else:
                cell.set_facecolor('#2A3744')
            cell.set_edgecolor('#4A5A6A')

            if j == 0:
                cell.set_text_props(color='white', fontsize=11, ha='left')
            elif j == 1:
                # Purchase column — RED
                cell.set_text_props(color='#E74C3C', fontweight='bold', fontsize=12)
            elif j == 2:
                # Rent column — GREEN
                cell.set_text_props(color='#2ECC71', fontweight='bold', fontsize=12)

        cell.set_edgecolor('#4A5A6A')
        cell.set_linewidth(0.8)

# ============================================
# TITLE
# ============================================
ax.set_title(
    "Table 9. Purchase vs. Rent — Full Utility Breakdown",
    fontsize=14, fontweight='bold', pad=15, color='white'
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig_count = 1


# Data for plotting
labels = ['Purchase', 'Rent']
utilities = [U_purchase, U_rent]
colors = ['#2ecc71', '#e74c3c']

# Create bar chart
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(labels, utilities, color=colors, edgecolor='black', width=0.3)

# Add value labels on top of bars
for bar, value in zip(bars, utilities):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f'{value:.2f}',
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold'
    )

# Formatting
ax.set_ylabel('Utility Value ($U$)', fontsize=11)
ax.set_ylim(-4, max(utilities) * 1.25)
ax.grid(axis='y', linestyle='--', alpha=0.6)
ax.axhline(y=0, color='black', linewidth=0.8)
plt.suptitle(
    f"Figure {fig_count}. Utility Comparison of Purchase and Rent", y=0.0001, fontsize=12
)
fig_count += 1
plt.tight_layout()
plt.show()